# LFPs plots in the parameters domain

In [7]:
SAVE_MEGA_CHART = False

Required files:
- TTL times
- TTL labels
- LFPs data (can be read using TTL times)
- Sleep/Wake labels


In [8]:
import numpy as np
import scipy.io
import glob

import os
import re

def find_files(root_folder, pattern):
    # Compile the regex pattern for matching
    regex = re.compile(pattern)
    
    matching_files = []
    for dirpath, dirnames, filenames in os.walk(root_folder):
        for filename in filenames:
            if regex.match(filename):
                file_path = os.path.join(dirpath, filename)
                matching_files.append(file_path)
                # print(f"File found: {file_path}")
    return matching_files

MOUSE_NAME = 'C9'
main_path = r'\\path\Pixel1\1_auditory_neuropixels_BarakH'
main_folders = glob.glob(main_path + f'\*_{MOUSE_NAME}_*')
SleepWakeLabels_fname = 'labels_All.mat'
TTL_times_fname = 'TTL_times_adj.txt'
TTL_labels_fname = 'TTL_labels.txt'
LFP_path_fname = r'.*\.lf\.bin$'

## Create MEGA chart

In [9]:
import pandas as pd
from Global_functions import read_file_by_time_steps
from pathlib import Path
import tqdm
from scipy import signal

def calc_gamma_power_per_trials(x, fs, freq_max, freq_min):
    spec_vec = []
    for trial in range(x.shape[0]):
        f, t, Sxx = signal.spectrogram(np.squeeze(x[trial,:]), fs,  scaling='spectrum')
        Sxx = Sxx[(f >= freq_min) & (f <= freq_max), :]
        t = t - 1
        f = f[(f >= freq_min) & (f <= freq_max)]
        baseline_spec = Sxx[:, t<-0.1]
        spec_vec.append(Sxx/np.expand_dims(np.mean(baseline_spec, axis=1), axis=1))
    induced_gamma_spec = np.mean(np.stack(spec_vec , axis=0), axis=0)
    mean_induced_gamma_power = np.mean(induced_gamma_spec[:, (t>0) & (t<2.5)])
    
    # return f, t, induced_gamma_spec, mean_induced_gamma_power
    return f, t, 10*np.log10(induced_gamma_spec), 10*np.log10(mean_induced_gamma_power)

def extract_params(data_array, t_vec):
    waveform = np.squeeze(np.mean(data_array[:, :], axis=0))
    amplitude = np.min(waveform)
    latency = t_vec[np.argmin(waveform)]
    ntrials = data_array.shape[0]
    _, _, _, induced_gamma = calc_gamma_power_per_trials(data_array, 1/(t_vec[1] - t_vec[0]), 150, 50)
    
    # return latency, amplitude, list(waveform), ntrials, induced_gamma
    return latency, amplitude, ntrials, induced_gamma

save_folder = main_path + f'\Analyzed_{MOUSE_NAME}'
os.makedirs(save_folder, exist_ok=True)
save_path = save_folder + f'\MEGA_CHART_{MOUSE_NAME}_gamma.pkl' 
SoundTypes = [101, 102, 400, 1301, 1401, 1402, 1403]
Injections = ['Saline', 'CNO', 'Saline', 'CNO', 'Saline', 'CNO']
chanList = list(range(120, 160, 5)) + list(range(80,120,5)) + list(range(160,200,5)) # List of channels to read

if os.path.exists(save_path):
    df = pd.read_pickle(save_path)
else:
    df = pd.DataFrame({"Date":[], "Injection":[], "SoundType":[], "Channel":[],
                   "Latency_All":[], "Amplitude_All":[], "Gamma_All":[], "NTrials_All":[],
                   "Latency_Sleep":[], "Amplitude_Sleep":[], "Gamma_Sleep":[], "NTrials_Sleep":[],
                   "Latency_Wake":[], "Amplitude_Wake":[], "Gamma_Wake":[],  "NTrials_Wake":[],
                   "Latency_First30min":[], "Amplitude_First30min":[], "Gamma_First30min":[], "NTrials_First30min":[],
                   "Latency_Last30min":[], "Amplitude_Last30min":[], "Gamma_Last30min":[], "NTrials_Last30min":[]})
    
display(df)

In [10]:
if SAVE_MEGA_CHART:
    for ind, main_folder in enumerate(main_folders):
        
        cur_date = os.path.splitext(os.path.basename(main_folder))[0].split('_')[0]
        if (cur_date != '20240317') and (cur_date != '20240318'):
            continue
        cur_injection = Injections[ind]
        
        LFP_path = find_files(main_folder, LFP_path_fname)[0]
        TTL_labels_path = find_files(main_folder, TTL_labels_fname)[0]
        TTL_times_path = find_files(main_folder, TTL_times_fname)[0]
        SleepWakeLabels_path = find_files(main_folder, SleepWakeLabels_fname)[0]
        
        TTL_times = np.loadtxt(TTL_times_path)
        TTL_labels = np.loadtxt(TTL_labels_path)
        SleepWakeLabels =  scipy.io.loadmat(SleepWakeLabels_path)
        SleepWakeLabels = np.squeeze(SleepWakeLabels['labels'])
        SleepWakeTimes = np.linspace(0, 2*60*60, len(SleepWakeLabels))  # the stop value is set by the time I used for manual scoring
        print(f'\033[91m Date \033[0m{cur_date} \033[91m Injection \033[0m{cur_injection}')
        
        ''' Go over All Sounds '''
        for SoundType in SoundTypes:
            if SoundType != 400:
                num_trials = np.sum(TTL_labels == SoundType)
                cur_SoundType_trialsTimes = TTL_times[TTL_labels == SoundType]
            else:
                num_trials = np.sum(np.bitwise_and(TTL_labels >= 400,TTL_labels <= 600))
                cur_SoundType_trialsTimes = TTL_times[np.bitwise_and(TTL_labels >= 400,TTL_labels <= 600)]
    
            print(f'\033[94m SoundType \033[0m{SoundType} - \033[96m Number of trials: \033[0m{num_trials}')
            cur_SoundType_trialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes > 1]
            
            ''' Find Sleep/Wake label '''
            closest_SleepWakeTimes = []
            for curTTLtime in cur_SoundType_trialsTimes:
                closest_SleepWakeTimes.append(np.argmin(np.abs(SleepWakeTimes - curTTLtime)))
            
            cur_SoundType_SleepWakeLabels = SleepWakeLabels[closest_SleepWakeTimes]
            
            ''' Extract Trial times '''
            All_TrialsTimes = cur_SoundType_trialsTimes
            First30min_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_trialsTimes<30*60]
            Last30min_TrialsTimes = cur_SoundType_trialsTimes[3*30*60<cur_SoundType_trialsTimes]
            Sleep_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_SleepWakeLabels == 3]
            Wake_TrialsTimes = cur_SoundType_trialsTimes[cur_SoundType_SleepWakeLabels == 2]
            print(f'Sleep/Wake/1st/last - {len(Sleep_TrialsTimes)}/{len(Wake_TrialsTimes)}/{len(First30min_TrialsTimes)}/{len(Last30min_TrialsTimes)}')
            
            ''' Read LFPs at channels All Trials '''
            Picked_TrialsTimes = All_TrialsTimes
            Trials_read_times_start = Picked_TrialsTimes - 1 # Take one second before Stimuli Onset
            tstep = 4.5 # Take 4.5 seconds after -1 from onset (3.5 seconds after stimuli)
            
            LFPs_by_trials, sr_LFP = read_file_by_time_steps(Path(LFP_path), Trials_read_times_start, tstep, chanList)
            min_length = min(arr.shape[1] for arr in LFPs_by_trials)
            LFPs_by_trials = [arr[:, :min_length] for arr in LFPs_by_trials]
            LFPs_by_trials = np.stack(LFPs_by_trials, axis=1)  # 'LFPs_by_trials' np array with shape (N_channels, N_trials, len_Trial)
            
            t_LFP = np.linspace(-1, tstep - 1, LFPs_by_trials.shape[2])
            Trials_label = list(range(0,len(Picked_TrialsTimes)))
                
            Trials_sleep = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Sleep_TrialsTimes])
            Trials_Wake = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Wake_TrialsTimes])
            Trials_first30min = np.array([np.where(Picked_TrialsTimes == j)[0] for j in First30min_TrialsTimes])
            Trials_last30min = np.array([np.where(Picked_TrialsTimes == j)[0] for j in Last30min_TrialsTimes])
            
            for ch_pick in chanList:
                print(f'Extract ch parameters ch: {ch_pick}')
                
                LFPs_by_trials_ch_pick = np.squeeze(LFPs_by_trials[chanList.index(ch_pick),:,:])
                LFPs_by_trials_ch_pick = LFPs_by_trials_ch_pick - np.expand_dims(np.mean(LFPs_by_trials_ch_pick, axis=1), axis=1)
                
                lat_All, amp_All, n_All, Gamma_All= extract_params(LFPs_by_trials_ch_pick[:, :], t_LFP)
                lat_Sleep, amp_Sleep, n_Sleep, Gamma_Sleep = extract_params(LFPs_by_trials_ch_pick[Trials_sleep, :], t_LFP)
                lat_Wake, amp_Wake, n_Wake, Gamma_Wake  = extract_params(LFPs_by_trials_ch_pick[Trials_Wake, :], t_LFP)
                lat_first30min, amp_first30min, n_first30min, Gamma_First30min = extract_params(LFPs_by_trials_ch_pick[Trials_first30min, :], t_LFP)
                lat_last30min, amp_last30min, n_last30min, Gamma_Last30min = extract_params(LFPs_by_trials_ch_pick[Trials_last30min, :], t_LFP)
                
                new_row = {"Date":cur_date, "Injection":cur_injection, "SoundType":SoundType, "Channel":ch_pick,
                       "Latency_All":lat_All, "Amplitude_All":amp_All, "Gamma_All":Gamma_All,  "NTrials_All":n_All,
                       "Latency_Sleep":lat_Sleep, "Amplitude_Sleep":amp_Sleep, "Gamma_Sleep":Gamma_Sleep, "NTrials_Sleep":n_Sleep,
                       "Latency_Wake":lat_Wake, "Amplitude_Wake":amp_Wake, "Gamma_Wake":Gamma_Wake,  "NTrials_Wake":n_Wake,
                       "Latency_First30min":lat_first30min, "Amplitude_First30min":amp_first30min, "Gamma_First30min":Gamma_First30min, "NTrials_First30min":n_first30min,
                       "Latency_Last30min":lat_last30min, "Amplitude_Last30min":amp_last30min, "Gamma_Last30min":Gamma_Last30min, "NTrials_Last30min":n_last30min}
                
                new_row_df = pd.DataFrame([new_row])
                df = pd.concat([df, new_row_df], ignore_index=True)
    
    df.to_pickle(save_path)

else:
    df = pd.read_pickle(save_path)

display(df)

## Plot parameters in different groupings
- Scatter Latency vs Amplitude - Grouping State
- Scatter Latency vs Amplitude - Grouping Date
- Scatter Latency vs Amplitude - Grouping SoundType
- Scatter Latency vs Amplitude - Grouping InjectionType
- Scatter Latency vs Amplitude - Grouping Channel

In [26]:
# %matplotlib qt
%matplotlib inline
import matplotlib.pyplot as plt

df_picked_days = df.copy()
# df_picked_days = df_picked_days[((df_picked_days['Date'] == '20240317') | (df_picked_days['Date'] == '20240318')) & (df_picked_days['Channel'].between(120, 160))]
df_picked_days = df_picked_days[(df_picked_days['Date'] == '20240317')]  ## Only Saline
# df_picked_days = df_picked_days[(df_picked_days['Date'] == '20240317') & (df_picked_days['Channel'].between(120, 160))]  ## Only Saline, Around ACx
# df_picked_days = df_picked_days[(df_picked_days['Date'] == '20240317') & (df_picked_days['Channel'].between(160, 200))]  ## Only Saline, Above ACx
# df_picked_days = df_picked_days[(df_picked_days['Date'] == '20240317') & (df_picked_days['Channel'].between(80, 120))]  ## Only Saline, Below ACx


nSleep = int(np.sum(df_picked_days['NTrials_Sleep'])/len(df_picked_days['NTrials_Sleep']))
nWake = int(np.sum(df_picked_days['NTrials_Wake'])/len(df_picked_days['NTrials_Wake']))
nAll = int(np.sum(df_picked_days['NTrials_All'])/len(df_picked_days['NTrials_All']))
nFirst30min = int(np.sum(df_picked_days['NTrials_First30min'])/len(df_picked_days['NTrials_First30min']))
nLast30min = int(np.sum(df_picked_days['NTrials_Last30min'])/len(df_picked_days['NTrials_Last30min']))
    
def plot_states(ax, ax2, df_picked_days, metric1, metric2, states):
    n = len(states)
    cmap = plt.get_cmap('viridis', n)
    colors = {label: cmap(i) for i, label in enumerate(states)}
    
    # Create the scatter plot
    for label, color in colors.items():
        ax.scatter(x=df_picked_days[f'{metric1}_{label}'], y=df_picked_days[f'{metric2}_{label}'], color=color, marker='.', s=250, alpha=1, label=label if label not in plt.gca().get_legend_handles_labels()[1] else "")
    for label, color in colors.items():
        ax.scatter(x=np.median(df_picked_days[f'{metric1}_{label}']), y=np.median(df_picked_days[f'{metric2}_{label}']), color=color, marker='*', s=600, edgecolors='red', linewidths=3)
    
    ax.legend(loc='lower right')
    if metric2 == 'Latency':
        ax.set_ylim([0, 0.03])
        ax.set_yticks(np.arange(0, 0.03, 0.005), np.arange(0, 30, 5))
        ax.set_ylabel(f'{metric2} [sec]')
    if metric2 == 'Gamma':
        ax.set_ylim([-0.5, 2.5])
        # ax.set_yticks(np.arange(0, 0.03, 0.001), np.arange(0, 30, 1))
        ax.set_ylabel(f'{metric2} [$\Delta$dB]')
    ax.set_xlim([-800, 0])
    ax.set_xlabel(f'{metric1} [$\mu$V]')
    
    n_metric1 = int(np.sum(df_picked_days[f'NTrials_{states[0]}'])/len(df_picked_days[f'NTrials_{states[0]}']))
    n_metric2 = int(np.sum(df_picked_days[f'NTrials_{states[1]}'])/len(df_picked_days[f'NTrials_{states[1]}']))
    n_All = int(np.sum(df_picked_days['NTrials_All'])/len(df_picked_days['NTrials_All']))
    states.append('')
    ax2.pie(np.array([n_metric1, n_metric2, n_All - n_metric2 - n_metric1]), autopct='%1.1f%%', labels=states)

f, ax = plt.subplots(figsize=(10,5), ncols=2, nrows=1)

# states = ['All', 'Sleep', 'Wake', 'First30min', 'Last30min']
# states = ['First30min', 'Last30min']
states = ['Sleep', 'Wake']

plot_states(ax[0], ax[1], df_picked_days, 'Amplitude', 'Gamma', states)

# mylabels = ['Sleep', 'Wake', 'Undefined']
# ax[1].pie(np.array([nSleep, nWake, nAll - nWake - nSleep]), autopct='%1.1f%%', labels = mylabels)
# f.suptitle(f"Latency vs. Amplitude vs. States")

In [12]:
# %matplotlib qt
%matplotlib inline
import matplotlib.pyplot as plt

df_plot = df_picked_days.copy()
# Use np.select to handle the range mappings
conditions = [
    df_plot['SoundType'].between(100, 200),
    df_plot['SoundType'].between(400, 600),
    df_plot['SoundType'].between(1300, 1400),
    df_plot['SoundType'].between(1400, 1500)
]
choices = ['Clicks', 'DRC', 'WhiteNoise', 'USV']

df_plot['SoundType'] = np.select(conditions, choices, default=df_plot['SoundType'])

def plot_soundtype_pie(ax, df_plot, picked_soundtype):
    df_soundtype = df_plot[df_plot['SoundType'] == picked_soundtype]
    nSleep = int(np.sum(df_soundtype['NTrials_Sleep'])/len(df_soundtype['NTrials_Sleep']))
    nWake = int(np.sum(df_soundtype['NTrials_Wake'])/len(df_soundtype['NTrials_Wake']))
    nAll = int(np.sum(df_soundtype['NTrials_All'])/len(df_soundtype['NTrials_All']))
    nFirst30min = int(np.sum(df_soundtype['NTrials_First30min'])/len(df_soundtype['NTrials_First30min']))
    nLast30min = int(np.sum(df_soundtype['NTrials_Last30min'])/len(df_soundtype['NTrials_Last30min']))
    mylabels = ['Sleep', 'Wake', 'Undefined']
    ax.pie(np.array([nSleep, nWake, nAll - nWake - nSleep]), autopct='%1.1f%%', labels = mylabels)
    ax.set_title(f'{picked_soundtype}')

f, ax = plt.subplots(figsize=(12,6), ncols=4, nrows=1)
plot_soundtype_pie(ax[0], df_plot, 'Clicks')
plot_soundtype_pie(ax[1], df_plot, 'DRC')
plot_soundtype_pie(ax[2], df_plot, 'WhiteNoise')
plot_soundtype_pie(ax[3], df_plot, 'USV')


In [23]:
# %matplotlib qt
%matplotlib inline
import matplotlib.pyplot as plt

def plot_field_from_df(ax, df, field_to_check, metric1, metric2, state1, state2):
    label_array = df[f'{field_to_check}']
    label_array_unique = np.unique(label_array)
    n = len(label_array_unique)
    cmap = plt.get_cmap('viridis', n)
    colors = {date: cmap(i) for i, date in enumerate(label_array_unique)}
    
    # Create the scatter plot
    for label, color in colors.items():
        subset = df[label_array == label]
        ax.scatter(x=subset[f'{metric1}_{state1}'], y=subset[f'{metric2}_{state1}'], color=color, marker='.', s=250, edgecolors='red', linewidths=1, label=label if label not in plt.gca().get_legend_handles_labels()[1] else "")
        ax.scatter(x=subset[f'{metric1}_{state2}'], y=subset[f'{metric2}_{state2}'], color=color, marker='.', s=250, edgecolors='green', linewidths=1)

    for label, color in colors.items():
        subset = df[label_array == label]
        x_vals = np.stack([subset[f'{metric1}_{state1}'], subset[f'{metric1}_{state2}']])
        y_vals = np.stack([subset[f'{metric2}_{state1}'], subset[f'{metric2}_{state2}']])
        ax.scatter(x=np.median(x_vals), y=np.median(y_vals), color=color, marker='*', s=600,  edgecolors='black', linewidths=2)

    
    ax.legend(loc='best')
    if metric2 == 'Latency':
        ax.set_ylim([0, 0.03])
        ax.set_yticks(np.arange(0, 0.03, 0.001), np.arange(0, 30, 1))
        ax.set_ylabel(f'{metric2} [sec]')
    if metric2 == 'Gamma':
        ax.set_ylim([-0.5, 2.5])
        # ax.set_yticks(np.arange(0, 0.03, 0.001), np.arange(0, 30, 1))
        ax.set_ylabel(f'{metric2} [$\Delta$dB]')
    ax.set_xlabel(f'{metric1} [$\mu$V]')
    ax.set_xlim([-800, 0])
    ax.set_title(f"{metric2} vs. {metric1} vs. {field_to_check}")

df_plot = df_picked_days.copy()

# Use np.select to handle the range mappings
conditions = [
    df_plot['SoundType'].between(100, 200),
    df_plot['SoundType'].between(400, 600),
    df_plot['SoundType'].between(1300, 1400),
    df_plot['SoundType'].between(1400, 1500)
]
choices = ['Clicks', 'DRC', 'WhiteNoise', 'USV']

df_plot['SoundType'] = np.select(conditions, choices, default=df_plot['SoundType'])

_, ax = plt.subplots(figsize=(5,5), ncols=1, nrows=1)
# plot_field_from_df(ax[0,0], df_plot, 'Injection', 'Amplitude', 'Gamma', 'Sleep', 'Wake')
# plot_field_from_df(ax[0,1], df_plot, 'Date', 'Amplitude', 'Gamma', 'Sleep', 'Wake')
# plot_field_from_df(ax[1,0], df_plot, 'SoundType', 'Amplitude', 'Gamma', 'Sleep', 'Wake')
plot_field_from_df(ax, df_plot, 'Channel', 'Amplitude', 'Gamma', 'Sleep', 'Wake')

_, ax = plt.subplots(figsize=(10,10), ncols=2, nrows=2)
plot_field_from_df(ax[0,0], df_plot, 'Injection', 'Amplitude', 'Gamma', 'Sleep', 'Wake')
plot_field_from_df(ax[0,1], df_plot, 'Date', 'Amplitude', 'Gamma', 'Sleep', 'Wake')
plot_field_from_df(ax[1,0], df_plot, 'SoundType', 'Amplitude', 'Gamma', 'Sleep', 'Wake')
plot_field_from_df(ax[1,1], df_plot, 'Channel', 'Amplitude', 'Gamma', 'Sleep', 'Wake')